In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Standalone colorbar for the |N error| -> B-factor spectrum used in render_error_structures.pml
(`spectrum b, blue_white_red, ..., minimum=VMIN, maximum=VMAX`). Kept separate from
compose_panel_figure.py so the scale can be regenerated on its own whenever VMIN/VMAX changes in
the .pml, without re-running the (slower) full panel composite.

Run:
    python draw_colorbar.py
"""
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# must match the `minimum=`/`maximum=` used in the `spectrum b, blue_white_red, ...` lines
VMIN, VMAX = 0.0, 5.0
UNIT = "ppm"
LABEL = "|predicted - ground truth| $^{15}$N error (ppm)"

# sampled directly from a rendered structure PNG (5506_1_error_on_structure.png) via the most
# common near-pure-blue / near-pure-red pixel values -- PyMOL's actual "blue"/"red" spectrum
# endpoints render as (4,4,255) / (255,4,4), not pure (0,0,255) / (255,0,0), so this matches the
# real renders exactly instead of approximating with web-pure colors.
PYMOL_WR = LinearSegmentedColormap.from_list("pymol_wr", ["#FFFFFF", "#FF0404"])


def draw_colorbar(vmin=VMIN, vmax=VMAX, unit=UNIT, label=LABEL, horizontal=True,
                   n_ticks=3, figsize=(6, 0.5), out_stem="colorbar"):
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    if not horizontal:
        gradient = gradient.T
        figsize = (figsize[1], figsize[0])

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(gradient, aspect="auto", cmap=PYMOL_WR,
              extent=(0, 1, 0, 1), origin="upper" if horizontal else "lower")

    tick_vals = np.linspace(vmin, vmax, n_ticks)
    ticks = (tick_vals - vmin) / (vmax - vmin)

    def fmt(v):
        s = f"{v:g}"   # "%g" drops trailing zeros (5 -> "5", 1.25 -> "1.25"), no banker's-rounding surprises
        return s

    tick_labels = [fmt(v) for v in tick_vals]

    if horizontal:
        ax.set_xticks(ticks)
        ax.set_xticklabels(tick_labels, fontsize=14, family="Arial")
        ax.set_yticks([])
        ax.set_xlabel(label, fontsize=14, family="Arial")
    else:
        ax.set_yticks(ticks)
        ax.set_yticklabels(tick_labels, fontsize=20, family="Arial")
        ax.yaxis.tick_right()
        ax.set_xticks([])
        ax.set_ylabel(label, fontsize=20, family="Arial", rotation=270, labelpad=24)
        ax.yaxis.set_label_position("right")


    for s in ax.spines.values():
        s.set_visible(False)

    fig.savefig(f"{out_stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{out_stem}.svg", bbox_inches="tight")
    print(f"wrote {out_stem}.png / .svg  (range {vmin}-{vmax} {unit})")
    return fig


if __name__ == "__main__":
    draw_colorbar(horizontal=False, out_stem="colorbar_vertical")
